# BM25·Dense 검색 결과의 RRF 결합

RRF(Reciprocal Rank Fusion, 순위 역수 결합)는 **여러 검색 결과의 순위를 합쳐 하나의 최종 순위를 만드는 방법**이다.

- Reciprocal Rank: 순위의 역수이다. 앞순위일수록 큰 값을 가진다.
- Fusion: 둘 이상의 검색 결과를 하나로 결합한다는 뜻이다.
- BM25: 단어 빈도·희귀도·문서 길이를 이용하는 키워드 기반 검색이다.
- Dense Retrieval: 질의와 문서를 임베딩 벡터로 바꿔 의미를 비교하는 검색이다.

BM25 score와 Dense 유사도는 계산 기준이 달라 숫자를 직접 더하기 어렵다. RRF는 각 검색기의 점수 대신 순위를 공통 기준으로 사용한다.

- raw score: BM25 또는 Dense 검색기가 만든 원래 점수이다.
- rank: raw score로 정렬한 문서의 위치이다.
- RRF score: 두 검색 순위의 역순위 점수를 더해 만든 결합 점수이다.
- rank_constant: 순위 차이가 RRF score에 미치는 영향을 조절하는 상수이다.

데이터 흐름은 `질의 → BM25·Dense 후보 순위 → RRF score → 상위 문서 ID → 공통 평가지표`이다.

## 평가 지표

- P@5(Precision at 5): 상위 5개 중 관련 문서의 비율이다.
- R@5(Recall at 5): 전체 관련 문서 중 상위 5개에서 찾은 비율이다.
- MRR@5(Mean Reciprocal Rank at 5): 상위 5개 안의 첫 관련 문서 순위 역수를 질의 전체에서 평균한 값이다.
- MAP@5(Mean Average Precision at 5): 관련 문서가 상위 5개의 앞쪽에 배치된 정도를 질의 전체에서 평균한 값이다.

## RRF 실행 패키지 준비

BM25와 Dense 후보 순위를 만들어 RRF 입력으로 사용하기 위해 관련 패키지를 준비한다.

- `%pip`: 현재 Jupyter 커널의 Python 환경에 패키지를 설치하는 명령이다.
- `pandas`: CSV를 DataFrame으로 읽고 검색 결과를 표로 만든다.
- `numpy`: 질의별 평가 지표를 배열로 만들고 평균한다.
- `rank_bm25`: BM25 score와 순위를 계산한다.
- `KoNLPy`: 한국어 형태소 분석기 `Okt`를 제공한다.
- `langchain`: LangChain의 공통 구성 요소와 실행 규칙을 제공한다.
- `langchain-openai`: OpenAI 임베딩을 LangChain에서 사용한다.
- `langchain-pinecone`: Pinecone을 LangChain Vector Store로 연결한다.
- `pinecone`: Pinecone 서비스와 통신하는 Python 패키지이다.
- `python-dotenv`: `.env` 파일을 환경 변수로 불러온다.
- `gdown`: Google Drive 파일을 ID로 내려받는다.


In [1]:
%pip install -U pandas numpy rank_bm25 konlpy langchain langchain-openai langchain-pinecone pinecone python-dotenv gdown

  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 실제 검색에 필요한 환경 설정

`.env`를 읽고 `01_indexing.ipynb`에서 문서를 저장할 때 사용한 설정을 같게 맞춘다.

- API key: OpenAI·Pinecone 요청 주체를 인증하는 비밀 값이다.
- `.env`: API key와 모델·index 설정을 코드 밖에 저장하는 파일이다.
- index: Pinecone에서 벡터를 저장하는 최상위 검색 단위이다.
- namespace: 하나의 index 안에서 벡터를 나누는 논리적 검색 영역이다.
- 임베딩 모델: 텍스트를 벡터로 변환하는 모델이다.
- 벡터 차원: 임베딩 배열의 길이이며 Pinecone index 설정과 같아야 한다.

이 설정은 뒤에서 Pinecone Vector Store를 연결할 때 사용한다.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=False)

PINECONE_INDEX_NAME = "adv-rag"
PINECONE_NAMESPACE = os.getenv("PINECONE_NAMESPACE", "")
OPENAI_EMBEDDING_MODEL = (
    os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small").strip()
    or "text-embedding-3-small"
)
PINECONE_INDEX_DIMENSION = int(os.getenv("PINECONE_INDEX_DIMENSION", "1536"))

## 문서와 질의 데이터 다운로드

BM25·Dense·RRF에 같은 문서·질의·정답 입력을 제공한다.

- CSV: 열을 쉼표로 구분한 표 형식의 텍스트 파일이다.
- `gdown`: Google Drive 파일 ID로 파일을 내려받는 명령이다.
- `-O`: 내려받은 파일의 저장 이름을 지정한다.


In [ ]:
!gdown 1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k -O documents.csv

!gdown 1dsn0pwkfzOUxiQ4MKDIvMM-CkxbYiSce -O queries.csv

## 문서·질의 구조 확인

DataFrame은 행과 열로 구성된 Pandas 표 객체이다. 두 CSV를 서로 다른 DataFrame으로 읽는다.

- `documents_df`: BM25가 검색할 문서 표이다.
- `doc_id`: 문서 식별자이다.
- `queries_df`: 검색 질의와 평가 정답 표이다.
- `query_id`: 세 검색 결과를 같은 질문끼리 연결하는 식별자이다.
- `query_text`: 검색기에 전달할 질의 문장이다.

### qrels

`qrels(Query Relevance Judgments)`는 질의별 관련 문서를 미리 기록한 **검색 평가용 정답표**이다.

- 현재 열: `relevant_doc_ids`이다.
- 저장 형식: `문서 ID=관련성 등급`이다.
- 예시: `D1=3;D4=1;D30=1`은 세 문서가 관련 문서라는 뜻이다.
- 주의: qrels는 검색 결과가 아니라 검색 성능을 평가하기 위해 미리 준비한 정답이다.


In [3]:
from pathlib import Path
import pandas as pd

documents_path = Path("documents.csv")
queries_path = Path("queries.csv")

# Pandas 이용해서 csv 읽어오기
documents_df = pd.read_csv(documents_path)
queries_df = pd.read_csv(queries_path)

display(documents_df.head())
display(queries_df.head())

,doc_id,title,content
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(..."
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기..."
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet..."
3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...
4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...


,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=3;D4=1;D30=1
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3


## RRF 입력 1 준비: BM25 검색기 만들기

RRF는 문서를 직접 검색하지 않는다. BM25와 Dense Retrieval이 각각 만든 문서 순위를 입력으로 받아 최종 순위를 만든다.

이 구간은 첫 번째 입력인 **BM25 문서 순위**를 만들기 위한 검색기를 준비한다.

- corpus: BM25가 검색할 전체 문서 모음이다.
- `Okt(Open Korean Text)`: 한국어 형태소 분석기이다.
- `morphs(str)`: 문자열을 형태소 목록으로 변환한다.
- `BM25Okapi`: 토큰화한 corpus를 색인하고 질의별 BM25 score를 계산한다.

여기서 색인은 Pinecone에 저장하는 작업이 아니다. BM25가 단어 빈도·희귀도·문서 길이를 계산할 수 있도록 메모리에 검색 정보를 준비하는 작업이다.

처리 흐름은 **문서 → 형태소 목록 → BM25 검색기 → BM25 문서 순위**이다.


In [6]:
from konlpy.tag import Okt
from rank_bm25 import BM25Okapi

okt = Okt() # 한국어 형태소 분석기

# 각 문서별 형태소 목록을 모아둔 2차원 list 생성
tokenized_documents = [
    okt.morphs(content)
    for content in documents_df["content"]
]

# 문서별 형태소 list를 색인하는 객체
# -> get_scores()를 이용해서 질의별 문서 점수를 계산 가능
bm25 = BM25Okapi(tokenized_documents)

### RRF 입력 1: BM25 문서 순위 함수

`bm25_search()`는 질의 하나를 받아 BM25 score가 높은 순서의 문서 ID 목록을 반환한다. 이 반환값이 RRF의 첫 번째 입력 순위가 된다.

- `query: str`: 검색 질의 문자열이다.
- `top_k: int`: 최대 반환 문서 수이다.
- `list[str]`: score 내림차순으로 정렬된 문서 ID 목록이다.
- `get_scores(tokens)`: 질의 형태소를 문서별 BM25 score 배열로 변환한다.

0보다 큰 score만 후보로 선택하므로 결과가 `top_k`보다 짧을 수 있다. 뒤의 결합 셀에서 이 함수를 호출해 `bm25_top20`을 만든다.


In [7]:
def bm25_search(query: str, top_k: int = 5) -> list[str]:
    # 1. 질의에도 문서와 같은 Okt 형태소 분석을 적용
    query_tokens = okt.morphs(query)

    # 2. document_scores[i]는 documents_df.iloc[i] 문서의 BM25 score
    document_scores = bm25.get_scores(query_tokens)

    # 3. list comprehension으로 score가 양수인 문서 행 번호만 모은다.
    positive_indices = [
        index
        for index, score in enumerate(document_scores)
        if score > 0
    ]

    # 4. 양수 score 후보에서 최대 top_k개 행 번호를 선택
    ranked_indices = sorted(
        positive_indices,
        key=lambda index: document_scores[index],
        reverse=True,
    )[:top_k]

    # 5. 행 번호를 RRF의 공통 결합 key인 doc_id 목록으로 변환
    ranked_doc_ids = [documents_df["doc_id"].iloc[index] for index in ranked_indices]
    return ranked_doc_ids


print( bm25_search("걸스데이 대표곡") )

['D3', 'D1', 'D9', 'D2', 'D19']


## RRF 입력 2 준비: Dense 검색기 연결하기

두 번째 입력은 Dense Retrieval이 만든 문서 순위이다. Dense Retrieval은 질의와 문서의 임베딩 벡터를 비교해 의미적으로 가까운 문서를 찾는다.

- Pinecone: 임베딩 벡터를 저장하고 유사도로 검색하는 관리형 Vector DB 서비스이다.
- `PineconeVectorStore`: Pinecone index를 LangChain의 Vector Store 인터페이스로 연결한다.
- `Document`: 본문 `page_content`와 부가 정보 `metadata`를 가진 LangChain 문서 객체이다.
- `metadata`: `doc_id`처럼 검색 결과를 원문과 연결하는 딕셔너리이다.
- cosine 유사도: 두 벡터의 방향이 비슷할수록 큰 값을 가진다.

이 구간에서는 `01_indexing.ipynb`에서 만든 Pinecone index를 Dense 검색기로 연결한다. 뒤의 결합 셀에서 `similarity_search()`를 호출해 RRF의 두 번째 입력인 `dense_top20`을 만든다.

처리 흐름은 **질의 → 임베딩 → Pinecone 검색 → Dense 문서 순위**이다.


In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

openai_embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL,
    dimensions=PINECONE_INDEX_DIMENSION,
)

vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=openai_embeddings,
    namespace=PINECONE_NAMESPACE,
)

## RRF 결합 준비: 두 문서 순위를 합치는 함수

앞의 두 구간은 BM25 문서 순위와 Dense 문서 순위를 준비한다. `rrf_fuse()`는 두 순위를 입력받아 문서별 역순위 점수를 합산한다.

$$
RRF(d)=\sum_{i=1}^{m}\frac{1}{k_{RRF}+rank_i(d)}
$$

- $d$: RRF score를 계산할 후보 문서이다.
- $i$: BM25·Dense와 같은 검색 목록 번호이다.
- $m$: 결합할 검색 목록 수이며 이 실습에서는 2이다.
- $rank_i(d)$: $i$ 번째 목록에서 문서 $d$의 순위이다. 해당 목록에 없으면 점수를 더하지 않는다.
- $k_{RRF}$: 순위 차이를 완화하는 상수이며 코드의 `rank_constant=60`이다.
- `top_k=20`: 각 검색기에서 가져올 최대 후보 수이며 $k_{RRF}$와 다른 값이다.
- 최종 5개: RRF score로 다시 정렬한 뒤 평가에 사용할 문서 수이다.

한 문서가 두 입력 순위의 상위에 반복해 등장하면 두 역순위 점수가 더해져 RRF score가 커진다.


In [9]:
# BM25와 Dense의 문서 ID 순위를 RRF score가 있는
# 결합 순위로 변환하는 함수
def rrf_fuse(
        bm25_ranked_ids: list[str],
        dense_ranked_ids: list[str],
        rank_constant: int = 60
) -> list[tuple[str, float]]:

    # doc_id별로 RRF score를 누적할 dict
    candidate_scores: dict[str, float] = {}

    # 1. BM25 순위의 역순위 점수를 doc_id 별로 누적하기
    for rank, doc_id in enumerate(bm25_ranked_ids, start=1):
        contribution = 1 / (rank_constant + rank)

        # 기존 score가 없는 경우 0, 있으면 누적
        candidate_scores[doc_id] = (
            candidate_scores.get(doc_id, 0.0) + contribution
        )

    # 2. Dense 순위의 역순위 점수를 doc_id 별로 누적하기
    for rank, doc_id in enumerate(dense_ranked_ids, start=1):
        contribution = 1 / (rank_constant + rank)

        # 기존 score가 없는 경우 0, 있으면 누적
        candidate_scores[doc_id] = (
            candidate_scores.get(doc_id, 0.0) + contribution
        )

    # 3. 정렬된 list[(doc_id, RRF score)] 반환
    ranked_items = sorted(
        candidate_scores.items(),

        # item : (doc_id, score)
        # -> item[1] == RRF score를 기준으로 정렬
        key=lambda item: item[1],

        reverse=True, # 내림차순
    )

    return ranked_items

## 두 입력 순위를 생성하고 RRF로 결합하기

이 구간에서 실제 질의별 문서 순위를 만들고 RRF에 전달한다.

1. **RRF 입력 1**: `bm25_search()`가 `bm25_top20` 문서 ID 순위를 만든다.
2. **RRF 입력 2**: `similarity_search()`가 반환한 `Document`에서 `dense_top20` 문서 ID 순위를 만든다.
3. **RRF 결합**: `rrf_fuse()`가 두 순위를 받아 `fused_top5`를 만든다.

BM25 후보가 20개보다 짧아도 오류가 아니다. RRF는 두 검색기가 반환한 후보만 다시 정렬하며 둘 다 놓친 문서를 새로 만들지는 않는다.


In [10]:
# 각 딕셔너리의 key: query_id
# 각 딕셔너리의 value: 해당 질의의 입력 순위 또는 RRF 결합 결과
bm25_candidates: dict[str, list[str]] = {}
dense_candidates: dict[str, list[str]] = {}
rrf_scored_results: dict[str, list[tuple[str, float]]] = {}
rrf_results: dict[str, list[str]] = {}


for _, row in queries_df.iterrows():
    query_id = row["query_id"]
    query_text = row["query_text"]

    # RRF 입력 1: BM25 score가 높은 순서의 doc_id를 최대 20개 가져온다.
    bm25_top20 = bm25_search(query_text, top_k=20)

    # RRF 입력 2: 질의를 벡터로 바꿔 의미가 가까운 Document를 최대 20개 찾는다.
    dense_documents = vector_store.similarity_search(
        query=query_text,
        k=20,
    )

    # Dense 검색 순서를 유지하며 각 Document의 metadata에서 doc_id만 꺼낸다.
    dense_top20 = [
        document.metadata["doc_id"]
        for document in dense_documents
    ]

    # 두 검색기가 만든 RRF 입력 순위를 query_id별로 저장한다.
    bm25_candidates[query_id] = bm25_top20
    dense_candidates[query_id] = dense_top20

    # RRF 결합: BM25 순위와 Dense 순위를 하나의 RRF 순위로 합친다.
    fused_top5 = rrf_fuse(
        bm25_top20,
        dense_top20,
        rank_constant=60,
    )[:5]

    # score가 필요한 결과와 문서 ID만 필요한 평가 결과를 각각 저장한다.
    rrf_scored_results[query_id] = fused_top5

    # tuple unpacking: (doc_id, score)에서 doc_id만 사용하고 score는 _로 받는다.
    rrf_results[query_id] = [
        doc_id
        for doc_id, _ in fused_top5
    ]

## 첫 번째 질의(Q1)의 BM25·Dense·RRF 순위 비교

`Q1`은 첫 번째 검색 질문을 구분하는 `query_id`이다. 순위나 점수와는 다른 식별값이다.

- 실제 질문: `제주도 올레길 트레킹 코스 추천`
- 평가용 관련 문서(qrels): `D1`, `D4`, `D30`

같은 Q1에 대해 BM25 순위, Dense 순위와 RRF 상위 5개를 차례대로 출력한다. 같은 문서가 두 입력 순위의 앞쪽에 있으면 RRF에서도 앞에 배치되는지 확인한다.


In [12]:
print(bm25_candidates["Q2"])
print(dense_candidates["Q2"])
print(rrf_results["Q2"])

['D13', 'D2', 'D10', 'D27', 'D9', 'D28']
['D2', 'D13', 'D27', 'D8', 'D12', 'D20', 'D21', 'D11', 'D6', 'D16', 'D15', 'D1', 'D5', 'D19', 'D10', 'D28', 'D9', 'D29', 'D24', 'D26']
['D13', 'D2', 'D27', 'D10', 'D9']


## Q1의 RRF score 확인

`rrf_scored_results`는 문서 ID와 RRF score를 함께 저장한다. 이 score는 BM25 score나 Dense 유사도를 더한 값이 아니라 두 순위에서 계산한 역순위 점수의 합이다.


In [13]:
q1_rrf_rows = []
for rank, (doc_id, score) in enumerate(
        rrf_scored_results["Q1"],
        start=1):

    q1_rrf_rows.append({
        "rank": rank,
        "doc_id": doc_id,
        "RRF_score": score,
    })

q1_rrf_score_df = pd.DataFrame(q1_rrf_rows)
q1_rrf_score_df

,rank,doc_id,RRF_score
0,1,D1,0.032787
1,2,D12,0.016129
2,3,D8,0.015873
3,4,D2,0.015625
4,5,D23,0.015385


## 전체 질의의 RRF 상위 5개 확인

- `rrf_results`: 질의별 RRF 상위 5개 문서 ID를 저장한 딕셔너리이다.
- `query_id`: 질문을 구분하는 ID이며 `rrf_results`의 key로 사용한다.
- RAG(Retrieval-Augmented Generation, 검색 증강 생성): 검색한 문서를 LLM 입력 문맥으로 넣어 답변을 생성하는 방식이다.

여기서는 검색 품질을 먼저 평가한다. 이후 RAG를 구성할 때는 선택된 문서 ID로 원문을 가져와 LLM에 전달한다.


In [15]:
pd.DataFrame(rrf_results)

,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,...,Q21,Q22,Q23,Q24,Q25,Q26,Q27,Q28,Q29,Q30
0,D1,D13,D3,D4,D5,D6,D25,D8,D9,D10,...,D21,D22,D23,D24,D25,D26,D27,D28,D29,D30
1,D12,D2,D1,D30,D24,D14,D7,D12,D27,D4,...,D23,D23,D19,D23,D14,D14,D28,D27,D7,D4
2,D8,D27,D19,D9,D28,D25,D30,D29,D20,D3,...,D27,D19,D26,D19,D6,D7,D29,D29,D30,D25
3,D2,D10,D9,D5,D15,D10,D14,D1,D13,D14,...,D19,D27,D24,D22,D10,D27,D30,D30,D22,D7
4,D23,D9,D17,D15,D8,D3,D6,D15,D16,D6,...,D18,D24,D22,D20,D7,D20,D26,D26,D28,D29


## qrels 정답 문자열 파싱

앞에서 확인한 qrels의 `relevant_doc_ids` 열을 평가 함수가 읽을 수 있는 딕셔너리로 바꾼다.

- 입력: `D6=3;D14=2` 형식의 문자열이다.
- 변환: 세미콜론으로 문서를 나누고 등급을 정수로 바꾼다.
- 출력: `{'D6': 3, 'D14': 2}` 형식의 딕셔너리이다.
- Binary relevance: 등급이 1 이상이면 관련 문서, 없으면 비관련 문서로 판단한다.


In [16]:
def parse_relevant(relevant_text: str) -> dict[str, int]:
    relevant_by_id: dict[str, int] = {}

    for pair in relevant_text.split(";"):
        doc_id, grade_text = pair.split("=")
        grade = int(grade_text)

        if grade > 0:
            relevant_by_id[doc_id] = grade

    return relevant_by_id

## Precision@k와 Recall@k 계산

- `k`: 검색기가 반드시 반환한 개수가 아니라 평가할 최대 순위이다.
- Precision@k: 상위 k개 자리 중 관련 문서가 차지한 비율이다.
- Recall@k: 전체 관련 문서 중 상위 k개 안에서 찾은 비율이다.

세 검색 방법에 같은 `k=5`를 적용한다. 결과가 5개보다 적으면 비어 있는 자리는 관련 문서를 찾지 못한 것으로 처리한다.


In [17]:
def precision_recall_at_k(
    predicted_ids: list[str],
    relevant_by_id: dict[str, int],
    k: int = 5,
) -> tuple[float, float]:
    # 1. 검색 결과의 앞 k개만 평가 대상으로 자른다.
    top_k_ids = predicted_ids[:k]

    # 2. 관련 문서 딕셔너리에 존재하는 ID의 개수를 센다.
    hit_count = 0
    for doc_id in top_k_ids:
        if doc_id in relevant_by_id:
            hit_count += 1

    # Precision@k: hit 수를 실제 반환 개수가 아닌 고정된 k로 나눈다.
    precision = hit_count / k

    # Recall@k: hit 수를 qrels에 등록된 전체 관련 문서 수로 나눈다.
    relevant_count = len(relevant_by_id)
    recall = hit_count / relevant_count if relevant_count else 0.0
    return precision, recall

## 첫 관련 문서의 Reciprocal Rank 계산

RR은 첫 관련 문서의 순위 역수이며 첫 문서가 정답이면 1.0, 관련 문서가 없으면 0.0이다. 여러 질의의 RR 평균이 MRR이며 RRF score와는 다른 평가값이다.


In [18]:
def reciprocal_rank(
    predicted_ids: list[str],
    relevant_by_id: dict[str, int],
) -> float:
    # start=1은 첫 검색 결과를 1위로 계산한다.
    for rank, doc_id in enumerate(predicted_ids, start=1):
        if doc_id in relevant_by_id:
            return 1 / rank

    # 관련 문서가 하나도 없을 때는 첫 정답 순위를 정의할 수 없으므로 0을 반환한다.
    return 0.0

## Average Precision@k 계산

AP@k는 상위 k개에서 관련 문서를 만날 때의 Precision을 평균해 여러 정답이 앞에 배치됐는지 측정한다. 모든 질의의 AP@k 평균이 MAP@k이다.


In [19]:
def average_precision_at_k(
    predicted_ids: list[str],
    relevant_by_id: dict[str, int],
    k: int = 5,
) -> float:
    precision_sum = 0.0
    relevant_seen = 0

    # 1. 상위 k개를 순서대로 보며 관련 문서가 나온 위치에서만 Precision을 누적한다.
    for rank, doc_id in enumerate(predicted_ids[:k], start=1):
        if doc_id in relevant_by_id:
            relevant_seen += 1
            precision_sum += relevant_seen / rank

    # 2. k 안에서 찾을 수 있는 최대 관련 문서 수를 분모로 사용한다.
    denominator = min(len(relevant_by_id), k)
    return precision_sum / denominator if denominator else 0.0

## 질의별 지표를 검색기 평균으로 집계

- `compute_metrics()`: 질의 하나의 P@k·R@k·RR·AP@k를 계산한다.
- `evaluate_all()`: 모든 질의의 지표를 모아 열별 평균을 계산한다.
- `metric_array`: 행은 질의, 열은 P@k·R@k·RR·AP@k인 2차원 배열이다.

BM25·Dense·RRF에 같은 정답과 같은 평가 함수를 적용해 검색 방식만 공정하게 비교한다.


In [22]:
import numpy as np

def compute_metrics(
    predicted_ids: list[str],
    relevant_by_id: dict[str, int],
    k: int = 5,
) -> tuple[float, float, float, float]:
    # 반환 tuple의 순서: (P@k, R@k, RR@k, AP@k)
    precision, recall = precision_recall_at_k(predicted_ids, relevant_by_id, k)
    rr = reciprocal_rank(predicted_ids[:k], relevant_by_id)
    ap = average_precision_at_k(predicted_ids, relevant_by_id, k)
    return precision, recall, rr, ap


def evaluate_all(
    method_results: dict[str, list[str]],
    query_table: pd.DataFrame,
    k: int = 5,
) -> dict[str, float]:
    # per_query_metrics: 질의별 (P, R, RR, AP) tuple을 저장할 목록
    per_query_metrics = []

    # 1. iterrows(): query_table을 (행 번호, Series) 단위로 반복한다.
    for _, row in query_table.iterrows():
        relevant_by_id = parse_relevant(row["relevant_doc_ids"])
        predicted_ids = method_results[row["query_id"]]

        # append(): 현재 질의의 지표 tuple을 목록 끝에 추가한다.
        per_query_metrics.append(
            compute_metrics(predicted_ids, relevant_by_id, k)
        )

    # 2. np.asarray(): tuple 목록을 shape=(질의 수, 4)인 2차원 배열로 바꾼다.
    # 행(axis=0): 질의
    # 열(axis=1): 0=P@k, 1=R@k, 2=RR@k, 3=AP@k
    metric_array = np.asarray(per_query_metrics, dtype=float)

    # [:, 열 번호]: 모든 행에서 해당 지표 열을 선택한다.
    # mean(): 선택한 열의 질의별 평균을 계산한다.
    return {
        "P@k": metric_array[:, 0].mean(),
        "R@k": metric_array[:, 1].mean(),
        "MRR@k": metric_array[:, 2].mean(),
        "MAP@k": metric_array[:, 3].mean(),
    }

## BM25·Dense·RRF 성능 비교

BM25·Dense 후보의 앞 5개와 RRF 상위 5개를 같은 정답·cutoff로 평가한다. RRF가 항상 모든 지표를 높이지는 않으므로 P@5, R@5, MRR@5, MAP@5를 함께 읽어 결합 효과를 판단한다.


In [23]:
# 1. BM25와 Dense도 RRF처럼 상위 5개만 평가하도록 cutoff를 통일한다.
bm25_results = {
    query_id: doc_ids[:5]
    for query_id, doc_ids in bm25_candidates.items()
}
dense_results = {
    query_id: doc_ids[:5]
    for query_id, doc_ids in dense_candidates.items()
}

# 2. 세 방법에 같은 queries_df와 k=5를 전달해 평균 지표를 계산한다.
bm25_metrics = evaluate_all(bm25_results, queries_df, k=5)
dense_metrics = evaluate_all(dense_results, queries_df, k=5)
rrf_metrics = evaluate_all(rrf_results, queries_df, k=5)

# 3. DataFrame의 행은 지표, 열은 검색 방식이 되도록 구성한다.
metrics_df = pd.DataFrame(
    {
        "Metric": ["P@5", "R@5", "MRR@5", "MAP@5"],
        "BM25": [
            bm25_metrics["P@k"],
            bm25_metrics["R@k"],
            bm25_metrics["MRR@k"],
            bm25_metrics["MAP@k"],
        ],
        "Dense": [
            dense_metrics["P@k"],
            dense_metrics["R@k"],
            dense_metrics["MRR@k"],
            dense_metrics["MAP@k"],
        ],
        "RRF": [
            rrf_metrics["P@k"],
            rrf_metrics["R@k"],
            rrf_metrics["MRR@k"],
            rrf_metrics["MAP@k"],
        ],
    }
)
metrics_df

,Metric,BM25,Dense,RRF
0,P@5,0.246667,0.260000,0.260000
1,R@5,0.883333,0.916667,0.916667
2,MRR@5,0.966667,0.983333,0.966667
3,MAP@5,0.849074,0.880741,0.857963
